In [50]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OrdinalEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,r2_score,mean_absolute_error

from sklearn.linear_model import LinearRegression 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [51]:

## Load Cleaned Data
df = pd.read_csv('../data/processed/student_placement_features.csv')

In [52]:
df.isnull().sum().sum()

np.int64(0)

In [53]:
df.duplicated().sum()

np.int64(0)

In [60]:
# Use the combined score, but drop the individual ingredients to prevent multicollinearity
features = [
    'cgpa', 'college_tier', 'coding_score', 'communication_score', 
    'aptitude_score', 'backlogs', 'resume_score', 'skill_score', 
    'experience_score', 'branch', 'academic_risk', 'student_segment'
]

X = df[features]
y = df['placed']            

In [61]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   cgpa                 9000 non-null   float64
 1   college_tier         9000 non-null   int64  
 2   coding_score         9000 non-null   float64
 3   communication_score  9000 non-null   float64
 4   aptitude_score       9000 non-null   float64
 5   backlogs             9000 non-null   int64  
 6   resume_score         9000 non-null   float64
 7   skill_score          9000 non-null   int64  
 8   experience_score     9000 non-null   float64
 9   branch               9000 non-null   object 
 10  academic_risk        9000 non-null   object 
 11  student_segment      9000 non-null   object 
dtypes: float64(6), int64(3), object(3)
memory usage: 843.9+ KB


In [62]:
# 1. Define the exact order for Ordinal variables (Lowest value to Highest value)
risk_categories = ['High Risk', 'Low Risk']
segment_categories = [
    '4. Low Skill + Low CGPA', 
    '3. Low Skill + High CGPA', 
    '2. High Skill + Low CGPA', 
    '1. High Skill + High CGPA'
]

# 2. Build the ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        # Format: (name, transformer, list_of_columns)
        ('ordinal', OrdinalEncoder(categories=[risk_categories, segment_categories]), ['academic_risk', 'student_segment']),
        ('nominal', OneHotEncoder(drop='first', sparse_output=False), ['branch'])
    ],
    remainder='passthrough' # This tells it to keep all our continuous numeric columns untouched
)

# 3. Apply the transformation
X_transformed = preprocessor.fit_transform(X)

# 4. Convert the resulting NumPy array back to a Pandas DataFrame for readability
# We use get_feature_names_out() so we don't lose our column names!
column_names = preprocessor.get_feature_names_out()
X_final = pd.DataFrame(X_transformed, columns=column_names)

# Optional: Clean up the generated column names to remove the prefixes added by ColumnTransformer
X_final.columns = X_final.columns.str.replace('ordinal__', '').str.replace('nominal__', '').str.replace('remainder__', '')

print("Final X shape:", X_final.shape)
print("\nFinal Columns ready for Machine Learning:")
print(X_final.columns.tolist())

Final X shape: (9000, 16)

Final Columns ready for Machine Learning:
['academic_risk', 'student_segment', 'branch_Civil', 'branch_ECE', 'branch_EEE', 'branch_IT', 'branch_Mechanical', 'cgpa', 'college_tier', 'coding_score', 'communication_score', 'aptitude_score', 'backlogs', 'resume_score', 'skill_score', 'experience_score']
